In [2]:
!nvidia-smi

Mon Jun  1 14:21:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import numpy as np
from numba import cuda, float32
import math


In [4]:
# Verify that a GPU is available
gpu_devices = cuda.detect()
if not gpu_devices:
    print("WARNING: No GPU detected. Please enable a GPU runtime in Colab.")
else:
    print("GPU is ready to go!")

Found 1 CUDA devices
id 0                Tesla T4                              [SUPPORTED]
                      Compute Capability: 7.5
                           PCI Device ID: 4
                              PCI Bus ID: 0
                                    UUID: GPU-ff0968cf-44ba-acb1-cc84-f8015c0fae7e
                                Watchdog: Disabled
             FP32/FP64 Performance Ratio: 32
Summary:
	1/1 devices are supported
GPU is ready to go!


In [5]:
@cuda.jit
def local_memory_kernel(d_out, d_in):
    # 1. Allocate Local Memory (Private to this specific thread)
    # Shape must be known at compile time.
    local_arr = cuda.local.array(shape=(5,), dtype=float32)

    # Get absolute thread index
    idx = cuda.grid(1)

    if idx < d_in.size:
        # Populate the local array with intermediate data
        for i in range(5):
            local_arr[i] = d_in[idx] * (i + 1.0)

        # Process data exclusively within this thread's local memory
        total = 0.0
        for i in range(5):
            total += local_arr[i]

        # Write final result back to global memory
        d_out[idx] = total

In [6]:
# Threads per block must be defined as a constant for shared memory allocation
THREADS_PER_BLOCK = 128

@cuda.jit
def shared_memory_kernel(d_out, d_in):
    # 1. Allocate Shared Memory (Accessible to all threads in THIS block)
    shared_arr = cuda.shared.array(shape=(THREADS_PER_BLOCK,), dtype=float32)

    # Get global thread index (for global memory) and local thread index (for shared memory)
    idx = cuda.grid(1)
    tx = cuda.threadIdx.x

    # 2. Load data from slow Global Memory into fast Shared Memory
    if idx < d_in.size:
        shared_arr[tx] = d_in[idx]
    else:
        shared_arr[tx] = 0.0 # Pad with zeros if out of bounds

    # 3. Synchronize!
    # Wait until all threads in the block have finished loading their data into shared_arr.
    cuda.syncthreads()

    # 4. Perform an operation using shared memory
    # Example: A simple 1D moving average (averaging neighbor values)
    if idx < d_in.size:
        # Grab left and right neighbors from shared memory (avoids extra global memory reads)
        left_val = shared_arr[tx - 1] if tx > 0 else 0.0
        right_val = shared_arr[tx + 1] if tx < THREADS_PER_BLOCK - 1 else 0.0
        center_val = shared_arr[tx]

        # Write result back to global memory
        d_out[idx] = (left_val + center_val + right_val) / 3.0

In [7]:
# 1. Prepare Host (CPU) Data
N = 1000000 # 1 Million elements
h_in = np.ones(N, dtype=np.float32) # Array of 1.0s
h_out_local = np.zeros(N, dtype=np.float32)
h_out_shared = np.zeros(N, dtype=np.float32)

# 2. Transfer Data to Device (GPU) Global Memory
d_in = cuda.to_device(h_in)
d_out_local = cuda.to_device(h_out_local)
d_out_shared = cuda.to_device(h_out_shared)

# 3. Configure Grid and Block Dimensions
blocks_per_grid = math.ceil(N / THREADS_PER_BLOCK)

print(f"Launching kernels with {blocks_per_grid} blocks of {THREADS_PER_BLOCK} threads...")


Launching kernels with 7813 blocks of 128 threads...


In [8]:
# 4. Launch Kernels
local_memory_kernel[blocks_per_grid, THREADS_PER_BLOCK](d_out_local, d_in)
shared_memory_kernel[blocks_per_grid, THREADS_PER_BLOCK](d_out_shared, d_in)

In [10]:
# 5. Wait for GPU to finish and copy data back to Host (CPU)
cuda.synchronize()
h_out_local = d_out_local.copy_to_host()
h_out_shared = d_out_shared.copy_to_host()

In [11]:
# 6. Verify Results
print("\n--- Results ---")
# Local Memory math check: 1.0 * (1+2+3+4+5) = 15.0
print(f"Local Memory Sample Output: {h_out_local[5]} (Expected: 15.0)")

# Shared Memory math check: (1.0 + 1.0 + 1.0) / 3.0 = 1.0
print(f"Shared Memory Sample Output: {h_out_shared[5]} (Expected: 1.0)")
print("Execution successful!")


--- Results ---
Local Memory Sample Output: 15.0 (Expected: 15.0)
Shared Memory Sample Output: 1.0 (Expected: 1.0)
Execution successful!
